### **1. Criação do Grafo**

In [1]:
import numpy as np
import csv

n = 10000
completo = False
file_name = f"grafo_{n}x{n}.csv"

matriz_adj = np.full((n, n), 1)

if(not completo):
  # Criando uma matriz de adjacências n x n simétrica (Grafo Não-Direcionado)
  for i in range(0, n):
      arestas = np.random.choice([0, 1], size = n - i)

      for j in range(i, n):
        x = arestas[j - i]
        matriz_adj[i, j] = x
        matriz_adj[j, i] = x

# Prevenção de laços no grafo
for i in range(n):
  matriz_adj[i, i] = 0

# Convertendo os valores da matriz para int de 8bits
matriz_adj = matriz_adj.astype(np.int8)

# Escrevendo o grafo gerado num arquivo .csv
with open(f"grafo_{n}x{n}.csv", 'w', newline='') as csvfile:
  grafo_writer = csv.writer(csvfile, delimiter=',')
  grafo_writer.writerows(matriz_adj)

### **2. Exploração do Grafo**

In [11]:
%%writefile bfs_openmp.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <omp.h>

// ===========================================================================//
//                              VARIÁVEIS GLOBAIS                             //
// ===========================================================================//

// Caminho do arquivo contendo a matriz original
char caminho[1024] = "/content/grafo_10000x10000.csv";

#define NUM_THREADS 2

int vertices_iniciais[NUM_THREADS] = {0, 9999};


// ===========================================================================//
//         CRIAÇÃO DE ESTRUTURAS AUXILIARES E SEUS RESPECTIVOS MÉTODOS        //
// ===========================================================================//

typedef struct celula{
    void *conteudo;
    struct celula *prox;
} Celula;

typedef struct{
    Celula *primeiro;
    Celula *ultimo;
} Fila;

void fila_push(Fila *F, void *conteudo_novo){
    if(F == NULL){
        printf("Fila inexistente!");
        return ;
    }

    Celula *nova = (Celula*)malloc(sizeof(Celula));
    if(nova == NULL){
        printf("Falha ao alocar memória para célula!\n");
        return ;
    }

    nova->conteudo = conteudo_novo;
    nova->prox = NULL;

    if(F->primeiro == NULL && F->ultimo == NULL){
        F->primeiro = nova;
        F->ultimo = nova;
    }
    else{
        F->ultimo->prox = nova;
        F->ultimo = nova;
    }
}

Celula* fila_pop(Fila *F){
    Celula *saida = NULL;
    if(F != NULL && F->primeiro != NULL && F->ultimo != NULL){
        saida = F->primeiro;

        if(F->primeiro == F->ultimo){
            F->primeiro = NULL;
            F->ultimo = NULL;
        }
        else{
            F->primeiro = F->primeiro->prox;
        }

        saida->prox = NULL;
    }

    return saida;
}


// ===========================================================================//
//          CRIAÇÃO DE ESTRUTURAS AUXILIARES PARA EXPLORAÇÃO NO GRAFO         //
// ===========================================================================//

typedef struct{
    int index_vertice;
    int dist;
    int index_pai;
} Vertice;

// ===========================================================================//
//          CRIAÇÃO DA ESTRUTURA DO GRAFO E SEUS RESPECTIVOS MÉTODOS          //
// ===========================================================================//

typedef struct{
    int num_vertices;
    int **matriz_adj;
} Grafo;

// Função que lê um grafo de um arquivo externo:
Grafo* instancia_grafo(char caminho[1024]){
    Grafo *g = NULL;

    FILE *fp = fopen(caminho, "r");

    if(!fp)
    printf("Não foi possível abrir o arquivo %s", caminho);

    else{
        // Inicializando estruturas auxiliares
        Fila *F = (Fila*)malloc(sizeof(Fila));
        if(F == NULL){
            printf("Erro de alocação de memória da fila!\n");
            return NULL;
        }

        F->primeiro = NULL;
        F->ultimo = NULL;

        // Criando o grafo
        g = (Grafo*)malloc(sizeof(Grafo));
        g->num_vertices = 0;
        g->matriz_adj = NULL;

        int n = 0;
        char *buffer = NULL;
        size_t tam_buffer = 0;

        // Lendo a primeira linha da matriz do arquivo
        if(getline(&buffer, &tam_buffer, fp) == -1){
            printf("Erro ao ler a primeira linha do arquivo.\n");
            fclose(fp);
            free(buffer);
            return NULL;
        }

        char *valor = strtok(buffer, ", ");
        while(valor != NULL){
            int *valor_int = (int*)malloc(sizeof(int));

            if(valor_int == NULL){
                printf("Erro ao alocar memória para um inteiro.\n");
                return NULL;
            }

            *valor_int = atoi(valor);
            void *vpvalor_int = valor_int;

            fila_push(F, vpvalor_int);
            valor = strtok(NULL, ", ");
            n++;
        }

        // Atualizando os atributos do grafo
        g->num_vertices = n;
        g->matriz_adj = (int**)malloc(n*sizeof(int*));

        if(g->matriz_adj == NULL){
            printf("Erro ao alocar memória para a matriz de adjacências.\n");
            return NULL;
        }

        // Instanciando a matriz de adjacência do grafo
        for(int i = 0; i < n; i++){
            g->matriz_adj[i] = (int*)calloc(n, sizeof(int));
            if(g->matriz_adj[i] == NULL){
                printf("Erro ao alocar memória na matriz de adjacências.\n");
                return NULL;
            }
        }

        // Preenchendo a primeira linha da matriz de adjacências
        for(int i = 0; i < n; i++){
            Celula *cel = fila_pop(F);
            int x = *(int *)cel->conteudo;
            g->matriz_adj[0][i] = x;

            free(cel->conteudo);
            free(cel);
        }

        free(F);

        // Lendo linhas restantes da matriz do arquivo
        int linha = 1, coluna = 0;
        valor = NULL;

        while(getline(&buffer, &tam_buffer, fp) != -1 && linha < n){
            coluna = 0;

            valor = strtok(buffer, ", ");

            while(valor != NULL && coluna < n){
                g->matriz_adj[linha][coluna] = atoi(valor);
                valor = strtok(NULL, ", ");

                coluna++;
            }

            linha++;
        }

        free(buffer);
        fclose(fp);
    }

    return g;
}

void print_grafo(Grafo *g){
    printf("Grafo:\n");
    printf("Numero de Vertices = %d\n", g->num_vertices);

    printf("Matriz de Adjacencia:\n");
    for(int i = 0; i < g->num_vertices; i++){
        for(int j = 0; j < g->num_vertices; j++){
            printf("%d ", g->matriz_adj[i][j]);
        }
        printf("\n");
    }
    printf("\n");
}


// ===========================================================================//
//                    ALGORITMO DE EXPLORAÇÃO EM GRAFOS (BFS)                 //
// ===========================================================================//

// Executa o BFS a partir de um vértice de origem. cinzas/pretos são
// compartilhados entre todas as "tarefas" (assim como no código em pthreads),
// de modo que os vértices já visitados por uma tarefa não são reexplorados
// pelas demais.
void BFS(Grafo *g, int index_tarefa, int index_vertice_origem,
          int *cinzas, int *pretos, Vertice ***resultados){

    // Inicialização dos vértices e estruturas auxiliares
    Vertice **vertices = (Vertice**)malloc(g->num_vertices*sizeof(Vertice*));
    if(vertices == NULL){
        printf("Erro ao alocar memória para o vetor de vértices.\n");
        return;
    }

    for(int i = 0; i < g->num_vertices; i++){
        vertices[i] = (Vertice*)malloc(sizeof(Vertice));
        if(vertices[i] == NULL){
            printf("Erro ao alocar memória para um vértice.\n");
            return;
        }
        vertices[i]->dist = -1;
        vertices[i]->index_pai = i;
        vertices[i]->index_vertice = i;
    }

    Fila *Q = (Fila*)malloc(sizeof(Fila));
    if(Q == NULL){
        printf("Erro ao alocar memória para a fila de vértices.\n");
        return;
    }

    Q->primeiro = NULL;
    Q->ultimo = NULL;

    int *origem_heap = (int*)malloc(sizeof(int));
    *origem_heap = index_vertice_origem;
    fila_push(Q, origem_heap);

    vertices[index_vertice_origem]->dist = 0;

    #pragma omp critical (acesso_visitados)
    {
        cinzas[index_vertice_origem] = 1;
    }

    #pragma omp critical (acesso_print)
    {
        printf("\nTarefa %d (thread %d) preparada para iniciar!\n",
               index_tarefa, omp_get_thread_num());
    }

    // Exploração do grafo em Largura
    while(Q->primeiro != NULL && Q->ultimo != NULL){
        Celula *u = fila_pop(Q);
        if(u == NULL){
            printf("Erro ao explorar o grafo pelo BFS.\n");
            return;
        }

        int u_index = *(int *)u->conteudo;

        for(int i = 0; i < g->num_vertices; i++){
            if(g->matriz_adj[u_index][i] != 0){
                int deve_expandir = 0;

                // Seção crítica: leitura/escrita do vetor global de visitados
                #pragma omp critical (acesso_visitados)
                {
                    if(cinzas[i] == 0){
                        cinzas[i] = 1;
                        deve_expandir = 1;
                    }
                }

                if(deve_expandir){
                    int *v_index = (int*)malloc(sizeof(int));
                    if(v_index == NULL){
                        printf("Erro ao alocar memoria para um inteiro.\n");
                        return;
                    }
                    *v_index = i;

                    fila_push(Q, v_index);

                    vertices[i]->index_pai = u_index;
                    vertices[i]->dist = vertices[u_index]->dist + 1;
                }
            }
        }

        #pragma omp critical (acesso_visitados)
        {
            pretos[u_index] = 1;
        }

        free(u->conteudo);
        free(u);
    }

    free(Q);

    resultados[index_tarefa] = vertices;
}

// ===========================================================================//
//                      IMPLEMENTAÇÃO PARALELA VIA OPENMP                     //
// ===========================================================================//

// Executa as NUM_THREADS tarefas de BFS distribuindo-as entre
// "num_threads_ativos" threads OpenMP. Passando num_threads_ativos = 1
// obtém-se a execução sequencial (baseline), usada para calcular
// Speedup e Eficiência.
Vertice*** BFS_multithread(Grafo *g, int num_threads_ativos, double *tempo_gasto){
    if(g == NULL || g->num_vertices <= 0){
        printf("Grafo inexistente!\n");
        return NULL;
    }

    Vertice ***resultados = (Vertice***)calloc(NUM_THREADS, sizeof(Vertice**));
    if(resultados == NULL){
        printf("Erro ao alocar memória para o vetor de resultados.\n");
        return NULL;
    }

    // Vetores globais de controle de visitados (compartilhados entre tarefas)
    int *pretos = (int*)calloc(g->num_vertices, sizeof(int));
    int *cinzas = (int*)calloc(g->num_vertices, sizeof(int));

    if(pretos == NULL || cinzas == NULL){
        printf("Erro ao alocar memória para o vetor de pretos ou cinzas.\n");
        return NULL;
    }

    // Medição do tempo de execução
    double inicio = omp_get_wtime();

    #pragma omp parallel for num_threads(num_threads_ativos) schedule(dynamic) \
        shared(g, resultados, cinzas, pretos, vertices_iniciais)
    for(int i = 0; i < NUM_THREADS; i++){
        BFS(g, i, vertices_iniciais[i], cinzas, pretos, resultados);
    }

    double fim = omp_get_wtime();

    *tempo_gasto = fim - inicio;

    printf("Threads utilizadas: %d | Tempo gasto: %f s\n",
           num_threads_ativos, *tempo_gasto);

    free(cinzas);
    free(pretos);

    return resultados;
}

void libera_resultados(Vertice ***resultados, Grafo *g){
    if(resultados == NULL) return;

    for(int i = 0; i < NUM_THREADS; i++){
        if(resultados[i] != NULL){
            for(int j = 0; j < g->num_vertices; j++){
                free(resultados[i][j]);
            }
            free(resultados[i]);
        }
    }
    free(resultados);
}

// ===========================================================================//
//                              PROGRAMA PRINCIPAL                            //
// ===========================================================================//

int main(){

    Grafo *g = instancia_grafo(caminho);

    if(g == NULL){
        printf("ERRO ao instanciar o grafo.\n");
        return 1;
    }

    printf("Busca em Largura (BFS) - OpenMP\n");

    // ---------------------------------------------------------------------
    // 1. Execução sequencial (1 thread) -> usada como baseline
    // ---------------------------------------------------------------------
    printf("\n--- Execucao Sequencial (1 thread) ---\n");
    double tempo_sequencial = 0.0;
    Vertice ***resultados_seq = BFS_multithread(g, 1, &tempo_sequencial);

    if(resultados_seq == NULL){
        printf("ERRO NA FUNÇÃO BFS_multithread (sequencial).\n");
        return 1;
    }
    libera_resultados(resultados_seq, g);

    // ---------------------------------------------------------------------
    // 2. Execução paralela (NUM_THREADS threads)
    // ---------------------------------------------------------------------
    printf("\n--- Execucao Paralela (%d threads) ---\n", NUM_THREADS);
    double tempo_paralelo = 0.0;
    Vertice ***resultados_par = BFS_multithread(g, NUM_THREADS, &tempo_paralelo);

    if(resultados_par == NULL){
        printf("ERRO NA FUNÇÃO BFS_multithread (paralela).\n");
        return 1;
    }
    /*
    for(int i = 0; i < NUM_THREADS; i++){
        printf("\nNavegacao feita pela Tarefa %d:\n", i);
        Vertice **vertices = resultados_par[i];
        for(int j = 0; j < g->num_vertices; j++){
            int index = vertices[j]->index_vertice;
            int dist = vertices[j]->dist;
            int pai = vertices[j]->index_pai;

            printf("Vertice (%d):\nIndex_pai: %d\nDist:%d\n\n", index, pai, dist);
        }
    }
    */
    libera_resultados(resultados_par, g);

    // ---------------------------------------------------------------------
    // 3. Cálculo e impressão das métricas de desempenho
    // ---------------------------------------------------------------------
    double speedup = tempo_sequencial / tempo_paralelo;
    double eficiencia = speedup / NUM_THREADS;

    printf("\n===================== METRICAS =====================\n");
    printf("Tempo (s)  - Sequencial (1 thread)  : %f\n", tempo_sequencial);
    printf("Tempo (s)  - Paralelo (%d threads)   : %f\n", NUM_THREADS, tempo_paralelo);
    printf("Speedup                             : %f\n", speedup);
    printf("Eficiencia                          : %f\n", eficiencia);
    printf("======================================================\n");

    // Liberação da matriz de adjacências do grafo
    for(int i = 0; i < g->num_vertices; i++){
        free(g->matriz_adj[i]);
    }
    free(g->matriz_adj);
    free(g);

    return 0;
}

Overwriting bfs_openmp.c


In [12]:
!gcc -fopenmp bfs_openmp.c -o bfs_openmp -O2
!./bfs_openmp

Busca em Largura (BFS) - OpenMP

--- Execucao Sequencial (1 thread) ---

Tarefa 0 (thread 0) preparada para iniciar!

Tarefa 1 (thread 0) preparada para iniciar!
Threads utilizadas: 1 | Tempo gasto: 1.260282 s

--- Execucao Paralela (2 threads) ---

Tarefa 0 (thread 0) preparada para iniciar!

Tarefa 1 (thread 1) preparada para iniciar!
Threads utilizadas: 2 | Tempo gasto: 1.286303 s

===================== METRICAS =====================
Tempo (s)  - Sequencial (1 thread)  : 1.260282
Tempo (s)  - Paralelo (2 threads)   : 1.286303
Speedup                             : 0.979771
Eficiencia                          : 0.489885
